# 52. Verifying gradients with `check_gradient`

**Objectives:**

- Build a `Minimizer` and call `Minimizer.check_gradient` at a point.
- Compare the JAX autodiff gradient against a central finite-difference gradient.
- Read the `GradientCheckResult` error-summary fields.

Run the cells in order in a fresh kernel. Masses are in GeV, invariants in GeV^2, and daughter
indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

## 1. Model and toy data

Same rho(770) + non-resonant model as the other lessons in this group, with a floatable
Cartesian coefficient on the non-resonant term.

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=60, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

data = generate_toy(
    model, 1500, parameters=truth, seed=2026,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
session = FitSession(model, data)
print(f"Generated {data.size} unweighted events; truth = {truth}")

Generated 1500 unweighted events; truth = {'NR.x': 0.55, 'NR.y': 0.3}


## 2. `check_gradient`

`FitSession.minimizer()` builds the low-level `Minimizer` over `session.objective` and
`session.parameters`. `Minimizer.check_gradient(values)` evaluates the NLL's JAX
`value_and_grad` at that point and compares it, parameter by parameter, to a central
finite-difference gradient (`step_scale` sets the relative step size). It returns a
`GradientCheckResult` and, by default, also prints a comparison table.

In [3]:
minimizer = session.minimizer()
point = {"NR.x": 0.35, "NR.y": 0.45}
check = minimizer.check_gradient(point)

parameter                     JAX    finite diff      abs err      rel err
NR.x                -4.854194e+02  -4.854194e+02    7.714e-08    1.589e-10
NR.y                 2.010040e+02   2.010040e+02    5.849e-08    2.910e-10
max abs error = 7.714e-08
max rel error = 2.910e-10


## 3. Reading the error summary

`GradientCheckResult` exposes the raw `jax_gradient`/`finite_difference_gradient` arrays plus
per-parameter `absolute_error`/`relative_error`, and the convenience properties
`max_absolute_error`/`max_relative_error`. Good agreement (small errors) confirms the automatic
gradient used by every fit in this package matches an independent finite-difference check.

In [4]:
print(f"max absolute error = {check.max_absolute_error:.3e}")
print(f"max relative error  = {check.max_relative_error:.3e}")
assert check.max_relative_error < 1e-4, "JAX and finite-difference gradients disagree"
print("JAX autodiff gradient matches the central finite-difference gradient.")

max absolute error = 7.714e-08
max relative error  = 2.910e-10
JAX autodiff gradient matches the central finite-difference gradient.


## Summary and exercises

1. Call `check_gradient` at the fitted minimum (after `session.fit(...)`) -- the gradient there
   should be close to zero in both the JAX and finite-difference versions.
2. Pass `print_table=False` to suppress the printed table and only use the returned
   `GradientCheckResult`.
3. Tighten or loosen `step_scale` and see how the finite-difference estimate's error changes.

Return to the [course guide](TUTORIALS.md).